In [2]:
import logging
import os

import xarray as xr
import numpy as np
import pcraster as pcr
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import resevoir_functions as rf

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

In [3]:
def estimate_discharge_for_environmental_flow(self, channelStorage):
    # z-score for 90th percentile; sets the environmental flow threshold
    z_score = 1.2816

    # Long-term variance of discharge using an online Welford algorithm
    varDischarge = self.m2tDischarge / \
                   pcr.max(1.,
                   pcr.min(self.maxTimestepsToAvgDischargeLong, self.timestepsToAvgDischarge) - 1.)
    stdDischarge = pcr.max(varDischarge**0.5, 0.0)

    # Floor at 10% of avg discharge to prevent flip-flop near the threshold
    minDischargeForEnvironmentalFlow = pcr.max(0.0, self.avgDischarge - z_score * stdDischarge)
    factor = 0.10
    minDischargeForEnvironmentalFlow = pcr.max(factor * self.avgDischarge, minDischargeForEnvironmentalFlow)
    minDischargeForEnvironmentalFlow = pcr.max(0.0, minDischargeForEnvironmentalFlow)

    return minDischargeForEnvironmentalFlow

#This function is most likely redundant as we have the net env flow from the netCDF files.

In [ ]:
#loading data
def load_data(file_path):
    try:
        data = xr.open_dataset(file_path)
        logger.info(f"Data loaded successfully from {file_path}")
        return data
    except Exception as e:
        logger.error(f"Error loading data from {file_path}: {e}")
        return None

file_path = os.path.join(os.getcwd(), 'Data', 'POINTDATA') + os.sep

latitude = 37.625072479248            #hardcoded coordinates for a dam used in Jennies model as well. Later the model should be improved to automatically get the coordinates of the dam we want to model
longitude = -122.458351135254


# Monthly averages; daily resolution can be re-run for a select region
discharge = load_data(file_path + "sos_resout_final_monthAvg_output.nc").sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})
inflow = load_data(file_path + "soswaterInflow_annuaTot_output.nc").sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})  # yearly totals
demand_tot = load_data(file_path + 'sosreduction_demand_dailyTot_output.nc').sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'})

Storage_validation = load_data(file_path + 'sosStor_check_dailyTot_output.nc').sel(lat=latitude, lon=longitude, method='nearest').to_dataframe().reset_index().drop(columns={'lon', 'lat'}) #also sets storage values at t=0

zz


2026-05-16 12:52:32,484 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/sos_resout_final_monthAvg_output.nc
2026-05-16 12:52:32,503 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/soswaterInflow_annuaTot_output.nc


2026-05-16 12:52:32,566 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/sosreduction_demand_dailyTot_output.nc
2026-05-16 12:52:32,679 INFO Data loaded successfully from /home/pablosnackbar/resevoir_study/Data/POINTDATA/sosStor_check_dailyTot_output.nc


sos_reservoir_outflow_end
time       lat       lon                                   
1979-01-31 43.958332 -124.958336                        NaN
                     -124.875000                        NaN
                     -124.791664                        NaN
                     -124.708336                        NaN
                     -124.625000                        NaN